# Notebook 03 — Detector Estadístico con Z-Score
Detecta anomalías numéricas en `data/metadata.csv` usando z-score por columna.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/metadata.csv')
print(f'Filas cargadas: {len(df)}')
df.head()

Filas cargadas: 50


,mesa_id,municipio,candidato_1,candidato_2,candidato_3,candidato_4,candidato_5,votos_nulos,total_votos,capacidad_mesa,tiene_anomalia,tipo_anomalia
0,Mesa_001,San Marcos,24,13,45,41,38,4,165,200,False,NaN
1,Mesa_002,La Esperanza,23,79,21,64,14,0,201,200,False,NaN
2,Mesa_003,Villa Nueva,49,36,40,26,19,0,170,200,True,votos_nulos_cero
3,Mesa_004,El Progreso,35,79,63,38,67,18,300,200,False,NaN
4,Mesa_005,Santa Rosa,45,10,30,64,53,8,210,200,False,NaN


In [2]:
cols_numericas = ['candidato_1','candidato_2','candidato_3',
                  'candidato_4','candidato_5','votos_nulos','total_votos']

# Z-score por columna
for col in cols_numericas:
    media = df[col].mean()
    std   = df[col].std()
    df[f'z_{col}'] = (df[col] - media) / std if std > 0 else 0.0

zcols = [f'z_{c}' for c in cols_numericas]
df['max_zscore'] = df[zcols].abs().max(axis=1)

UMBRAL_Z = 2.5
df['sospechosa_zscore'] = df['max_zscore'] > UMBRAL_Z

In [3]:
print(f'=== Top 10 mesas por z-score (umbral={UMBRAL_Z}) ===')
top10 = df.nlargest(10, 'max_zscore')[['mesa_id','municipio','max_zscore','sospechosa_zscore','tiene_anomalia','tipo_anomalia']]
print(top10.to_string(index=False))

=== Top 10 mesas por z-score (umbral=2.5) ===
 mesa_id     municipio  max_zscore  sospechosa_zscore  tiene_anomalia         tipo_anomalia
Mesa_020         Petén    3.351153               True            True participacion_extrema
Mesa_045  Jacaltenango    2.849107               True            True participacion_extrema
Mesa_016    Chiquimula    2.052506              False           False                   NaN
Mesa_024         Cobán    1.995333              False            True      total_incorrecto
Mesa_046      Barillas    1.986817              False           False                   NaN
Mesa_039       Sibinal    1.938963              False           False                   NaN
Mesa_004   El Progreso    1.915158              False           False                   NaN
Mesa_008        Sololá    1.913479              False           False                   NaN
Mesa_012 Suchitepéquez    1.833699              False           False                   NaN
Mesa_023     San Pedro    1.823814

In [4]:
print('=== Métricas de detección ===')
vp = ((df['sospechosa_zscore'] == True)  & (df['tiene_anomalia'] == True)).sum()
fp = ((df['sospechosa_zscore'] == True)  & (df['tiene_anomalia'] == False)).sum()
fn = ((df['sospechosa_zscore'] == False) & (df['tiene_anomalia'] == True)).sum()
vn = ((df['sospechosa_zscore'] == False) & (df['tiene_anomalia'] == False)).sum()

print(f'Verdaderos Positivos (VP): {vp}')
print(f'Falsos Positivos    (FP): {fp}')
print(f'Falsos Negativos    (FN): {fn}')
print(f'Verdaderos Negativos(VN): {vn}')
precision = vp / (vp + fp) if (vp + fp) > 0 else 0
recall    = vp / (vp + fn) if (vp + fn) > 0 else 0
print(f'Precisión: {precision:.2f} | Recall: {recall:.2f}')

=== Métricas de detección ===
Verdaderos Positivos (VP): 2
Falsos Positivos    (FP): 0
Falsos Negativos    (FN): 8
Verdaderos Negativos(VN): 40
Precisión: 1.00 | Recall: 0.20


In [5]:
print('=== Columna con más alertas ===')
alertas_por_col = {}
for col, zcol in zip(cols_numericas, zcols):
    alertas_por_col[col] = (df[zcol].abs() > UMBRAL_Z).sum()

alertas_series = pd.Series(alertas_por_col).sort_values(ascending=False)
print(alertas_series.to_string())
print(f'\nColumna líder: {alertas_series.idxmax()} ({alertas_series.max()} alertas)')

=== Columna con más alertas ===
votos_nulos    2
candidato_2    0
candidato_1    0
candidato_3    0
candidato_4    0
candidato_5    0
total_votos    0

Columna líder: votos_nulos (2 alertas)


In [6]:
df.to_csv('../data/resultados_zscore.csv', index=False)
print('Exportado: ../data/resultados_zscore.csv')

Exportado: ../data/resultados_zscore.csv
